# 06 — Proximal, Distal & Departure

## Proximal & Distal levels

Each confirmed formation defines a **zone box** with two boundary levels:

| Zone type | Proximal (near side — re-entry trigger) | Distal (far side — stop-loss reference) |
|-----------|----------------------------------------|----------------------------------------|
| **Demand** (RBR / DBR) | `base_high` — top of the base | `base_low` — bottom of the base |
| **Supply** (DBD / RBD) | `base_low` — bottom of the base | `base_high` — top of the base |

**Why this geometry?**
- A demand zone was left *upward* — when price eventually returns it comes from above, hitting the **top of the base first**.
- A supply zone was left *downward* — price returns from below, hitting the **bottom of the base first**.

The proximal line is the level traders watch for re-entry; the distal line is the stop reference.

---

## Departure strength

A formation alone is not enough — price must **leave with conviction**. We scan the `DEPARTURE_CANDLES` bars immediately after the base and measure peak excursion beyond the proximal level:

$$\text{departure (demand)} = \max(\text{high}_{be+1 \dots be+N}) - \text{proximal}$$
$$\text{departure (supply)} = \text{proximal} - \min(\text{low}_{be+1 \dots be+N})$$

Then **both** gates must pass:

| Gate | Formula | Threshold |
|------|---------|----------|
| ATR multiple | $\text{departure} / \overline{\text{ATR}} \geq 1.5$ | Volatility-adjusted strength |
| Zone ratio | $\text{departure} / \text{zone\_width} \geq 1.0$ | Move large relative to the zone itself |

The zone-ratio gate catches cases where a move looks strong in ATR terms but barely clears the zone height.

---

| Constant | Default | Meaning |
|---|---|---|
| `DEPARTURE_CANDLES` | 3 | Bars after base end to scan |
| `DEPARTURE_ATR_MIN` | 1.5 | Minimum departure / ATR |
| `DEPARTURE_RATIO_MIN` | 1.0 | Minimum departure / zone_width |

## 1. Setup & load data

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from IPython.display import display

from utils.data_loader import load_all_timeframes
from utils.models import CandlePrimitives, add_atr
from utils.base_detector import detect_bases
from utils.legs_formation import detect_formations
from utils.zone_detector import detect_zones
from utils.config import (
    COLOR_BULL, COLOR_BEAR,
    CHART_BG as BG, CHART_GRID as GRID,
    DEPARTURE_CANDLES
)

SYMBOL = "USDJPY=X"

raw = load_all_timeframes(SYMBOL, align=False)
data = {tf: add_atr(CandlePrimitives.enrich_dataframe(df)) for tf, df in raw.items()}
common_start = max(df.index[0] for df in raw.values())
data = {tf: df[df.index >= common_start] for tf, df in data.items()}

print(f"Loaded {len(data)} timeframes: {list(data.keys())}")
for tf, df in data.items():
    print(f"  {tf:4s}  {len(df):5d} rows  {df.index[0].date()} → {df.index[-1].date()}")

Loaded 4 timeframes: ['1wk', '1d', '4h', '1h']
  1wk     104 rows  2024-06-09 → 2026-05-31
  1d      581 rows  2024-06-06 → 2026-06-05
  4h     3174 rows  2024-06-06 → 2026-06-05
  1h    12259 rows  2024-06-05 → 2026-06-05


## 2. Run the full pipeline

In [2]:
# bases → formations → zones (with departure gate)
results = {}
for tf, df in data.items():
    passed_bases, _     = detect_bases(df)
    formations          = detect_formations(df, passed_bases)
    passed_zones, rej   = detect_zones(df, formations)
    results[tf] = {
        "df":           df,
        "formations":   formations,
        "passed_zones": passed_zones,
        "rejected":     rej,
    }

# Summary table
rows = []
for tf, r in results.items():
    pz = r["passed_zones"]
    rj = r["rejected"]
    n_demand = sum(1 for z in pz if z["zone_type"] == "demand")
    n_supply = sum(1 for z in pz if z["zone_type"] == "supply")
    rows.append({
        "timeframe":        tf,
        "formations_in":    len(r["formations"]),
        "zones_passed":     len(pz),
        "demand_zones":     n_demand,
        "supply_zones":     n_supply,
        "rejected_dep":     len(rj),
        "pass_rate":        f"{len(pz)/len(r['formations']):.0%}" if r["formations"] else "—",
    })

summary = pd.DataFrame(rows).set_index("timeframe")
display(summary)

,formations_in,zones_passed,demand_zones,supply_zones,rejected_dep,pass_rate
timeframe,,,,,,
1wk,18,2,1,1,16,11%
1d,131,36,18,18,95,27%
4h,637,153,79,74,484,24%
1h,2357,493,229,264,1864,21%


## 3. Inspect — daily zones

In [3]:
TF = "1d"
df_1d = results[TF]["df"]

detail_rows = []
for z in results[TF]["passed_zones"] + results[TF]["rejected"]:
    bs, be = z["start"], z["end"]
    detail_rows.append({
        "date_start":   df_1d.index[bs].date(),
        "date_end":     df_1d.index[be].date(),
        "formation":    z["formation"],
        "zone_type":    z["zone_type"],
        "proximal":     z["proximal"],
        "distal":       z["distal"],
        "zone_width":   z["zone_width"],
        "dep_atr":      z["dep_atr"],
        "dep_ratio":    z["dep_ratio"],
        "passed":       z["passed"],
    })

detail = pd.DataFrame(detail_rows).sort_values("date_start")
if detail.empty:
    print(f"No zones found for {TF}")
else:
    display(detail.to_string(index=False))

'date_start   date_end formation zone_type   proximal     distal  zone_width  dep_atr  dep_ratio  passed\n2024-06-16 2024-06-19       RBR    demand 158.229004 157.149994     1.07901    2.875      1.553   False\n2024-06-24 2024-06-25       RBR    demand 159.934006 158.740005     1.19400    2.029      1.129   False\n2024-06-27 2024-06-28       RBR    demand 161.281998 160.257996     1.02400    0.658      0.450   False\n2024-07-08 2024-07-08       DBD    supply 160.257996 161.119995     0.86200    3.988      3.305    True\n2024-07-12 2024-07-16       DBD    supply 157.151001 159.388000     2.23700    1.755      0.799   False\n2024-07-19 2024-07-19       RBD    supply 156.951996 157.862000     0.91000    1.204      1.586   False\n2024-07-22 2024-07-22       RBD    supply 156.279999 157.613007     1.33301    3.788      3.260    True\n2024-07-29 2024-07-30       RBD    supply 152.404999 155.218994     2.81400    4.348      2.130    True\n2024-08-01 2024-08-01       DBD    supply 148.505997 1

## 4. Visualize — all 4 timeframes

In [4]:
import plotly.graph_objects as go

PLOT_MONTHS = 2

FILL_PASSED = {"demand": "rgba(38,166,154,0.15)",  "supply": "rgba(239,83,80,0.15)"}
FILL_REJ    = {"demand": "rgba(38,166,154,0.06)",  "supply": "rgba(239,83,80,0.06)"}
EDGE_COLOR  = {"demand": "#26a69a",                "supply": "#ef5350"}


def plot_zones(tf: str, r: dict) -> None:
    df         = r["df"]
    all_zones  = r["passed_zones"] + r["rejected"]

    cutoff  = df.index[-1] - pd.DateOffset(months=PLOT_MONTHS)
    df_plot = df[df.index >= cutoff].copy()
    offset  = len(df) - len(df_plot)

    vis_zones = [z for z in all_zones if z["end"] >= offset]

    n_passed_all = len(r["passed_zones"])
    n_rej_all    = len(r["rejected"])

    fig = go.Figure()

    # Candlestick
    fig.add_trace(go.Candlestick(
        x=df_plot.index,
        open=df_plot["open"], high=df_plot["high"],
        low=df_plot["low"],   close=df_plot["close"],
        increasing_line_color=COLOR_BULL,
        decreasing_line_color=COLOR_BEAR,
        name="Price",
    ))

    for z in vis_zones:
        bs_plot = max(z["start"] - offset, 0)
        be_plot = min(z["end"]   - offset, len(df_plot) - 1)
        zt      = z["zone_type"]
        passed  = z["passed"]

        x0 = df_plot.index[bs_plot]
        x1 = df_plot.index[be_plot]

        # Zone base cluster shading
        fig.add_vrect(
            x0=x0, x1=x1,
            fillcolor=FILL_PASSED[zt] if passed else FILL_REJ[zt],
            line_color=EDGE_COLOR[zt],
            line_width=1,
            line_dash="solid" if passed else "dot",
            opacity=1.0,
        )

        # Proximal & distal horizontal lines extending into departure window
        ext_iloc = min(z["end"] - offset + DEPARTURE_CANDLES + 1, len(df_plot) - 1)
        x_ext   = df_plot.index[ext_iloc]

        fig.add_shape(
            type="line", x0=x0, x1=x_ext,
            y0=z["proximal"], y1=z["proximal"],
            line=dict(color=EDGE_COLOR[zt], width=1, dash="dash"),
        )
        fig.add_shape(
            type="line", x0=x0, x1=x_ext,
            y0=z["distal"], y1=z["distal"],
            line=dict(color=EDGE_COLOR[zt], width=1, dash="dot"),
        )

        # Annotation: formation + dep_atr / dep_ratio + pass/fail
        status = "✓" if passed else "✗"
        fig.add_annotation(
            x=x0, y=z["proximal"],
            text=f"{z['formation']} {status}<br>atr={z['dep_atr']} r={z['dep_ratio']}",
            showarrow=False,
            font=dict(size=8, color=EDGE_COLOR[zt]),
            xanchor="left", yanchor="bottom",
        )

    fig.update_layout(
        title=(
            f"{SYMBOL} {tf} — Zones (last {PLOT_MONTHS} months shown)  "
            f"full history: {n_passed_all} passed, {n_rej_all} rejected by departure"
        ),
        xaxis_rangeslider_visible=False,
        template="plotly_dark",
        paper_bgcolor=BG,
        plot_bgcolor=BG,
        xaxis=dict(gridcolor=GRID),
        yaxis=dict(gridcolor=GRID, title="Price"),
        height=500,
    )
    fig.show()


for tf, r in results.items():
    plot_zones(tf, r)

## 5. What's next

| Notebook | Topic | Status |
|---|---|---|
| `07_verify_all_scenarios.ipynb` | End-to-end pipeline — all filters together | ⏳ next |

We now have the complete zone object:
- **Formation** (RBR / DBD / DBR / RBD) and zone type (demand / supply)
- **Proximal / distal** boundary levels
- **Departure** confirmation that price left with institutional intent

NB07 wires all stages into one end-to-end pipeline and verifies it against all known scenarios.